# Eksperyment: Model Slicing dla diagnostyki modelu

## Cel eksperymentu
Gdy patrzymy tylko na ogólne accuracy modelu (np. 64%), tracimy informację o tym, **gdzie konkretnie** model się myli. Może świetnie radzić sobie z Hard Court i kompletnie zawalać Grand Slamy.

**Model Slicing** to technika z artykułu *GuideAI25_2 — Model Slicing for Responsible AI* (Godfrey et al., VLDB 2025). Polega na:
1. Podzieleniu zbioru testowego na **slice'y** — semantyczne podgrupy (np. mecze Bo5, mecze QF, mecze leworęczny-vs-praworęczny).
2. Policzeniu accuracy **per slice**.
3. Znalezieniu slice'ów, gdzie model wyraźnie niedomaga względem średniej.

## Dlaczego to ma znaczenie
- Slice'y wskazują **konkretne miejsca do poprawy** (cechami, treningiem, danymi).
- Wilson Confidence Interval mówi, czy underperformance jest **istotny statystycznie**, czy to szum (przy support=5).
- Log Loss i Brier Score per slice pokazują, czy model jest tylko niedokładny, czy też **źle skalibrowany** (zbyt pewny błędnych predykcji).

In [1]:
import itertools
import os
import runpy
from pathlib import Path

import numpy as np
import pandas as pd

MIN_SUPPORT = 5
MAX_SLICE_DEGREE = 2
UNDERPERFORMANCE_GAP = -0.05
TOP_N = 12
WILSON_Z = 1.96
LOG_EPS = 1e-15

## 1. Uruchomienie baseline pipeline

Reużywamy istniejący pipeline `tennis_model.py` — bez zmian. Funkcja `runpy.run_path` uruchamia skrypt i zwraca jego namespace, z którego wyciągniemy:
- `df_test_raw` — surowe dane testowe z atrybutami meczu (surface, round, ranking)
- `winner_perspective` — predykcje z perspektywy zwycięzcy
- `match_accuracy` — overall match accuracy do sanity check

In [2]:
BASE_SCRIPT = Path("../src/tennis_model.py").resolve()

namespace = runpy.run_path(str(BASE_SCRIPT))

df_test_raw = namespace["df_test_raw"].copy()
winner_perspective = namespace["winner_perspective"].copy()
reported_match_accuracy = float(namespace["match_accuracy"])

print(f"Liczba meczow testowych: {len(df_test_raw)}")
print(f"Match accuracy baseline: {reported_match_accuracy:.4f}")

Dane główne (2025): 2647 meczów
Zaladowano dane historyczne (C:\Users\stasi\PycharmProjects\TenisPredictionModel\data\sample_data\atp_matches_2001.csv): 2920 meczow
Zaladowano dane historyczne (C:\Users\stasi\PycharmProjects\TenisPredictionModel\data\sample_data\atp_matches_2002.csv): 2803 meczow
Zaladowano dane historyczne (C:\Users\stasi\PycharmProjects\TenisPredictionModel\data\sample_data\atp_matches_2003.csv): 2775 meczow
Zaladowano dane historyczne (C:\Users\stasi\PycharmProjects\TenisPredictionModel\data\sample_data\atp_matches_2004.csv): 2835 meczow
Zaladowano dane historyczne (C:\Users\stasi\PycharmProjects\TenisPredictionModel\data\sample_data\atp_matches_2005.csv): 2874 meczow
Zaladowano dane historyczne (C:\Users\stasi\PycharmProjects\TenisPredictionModel\data\sample_data\atp_matches_2006.csv): 2862 meczow
Zaladowano dane historyczne (C:\Users\stasi\PycharmProjects\TenisPredictionModel\data\sample_data\atp_matches_2007.csv): 2776 meczow


Zaladowano dane historyczne (C:\Users\stasi\PycharmProjects\TenisPredictionModel\data\sample_data\atp_matches_2008.csv): 2715 meczow
Zaladowano dane historyczne (C:\Users\stasi\PycharmProjects\TenisPredictionModel\data\sample_data\atp_matches_2009.csv): 2705 meczow
Zaladowano dane historyczne (C:\Users\stasi\PycharmProjects\TenisPredictionModel\data\sample_data\atp_matches_2010.csv): 2665 meczow
Zaladowano dane historyczne (C:\Users\stasi\PycharmProjects\TenisPredictionModel\data\sample_data\atp_matches_2011.csv): 2663 meczow
Zaladowano dane historyczne (C:\Users\stasi\PycharmProjects\TenisPredictionModel\data\sample_data\atp_matches_2012.csv): 2646 meczow
Zaladowano dane historyczne (C:\Users\stasi\PycharmProjects\TenisPredictionModel\data\sample_data\atp_matches_2013.csv): 2590 meczow
Zaladowano dane historyczne (C:\Users\stasi\PycharmProjects\TenisPredictionModel\data\sample_data\atp_matches_2014.csv): 2562 meczow
Zaladowano dane historyczne (C:\Users\stasi\PycharmProjects\TenisPred

Zaladowano dane historyczne (C:\Users\stasi\PycharmProjects\TenisPredictionModel\data\sample_data\atp_matches_2016.csv): 2830 meczow
Zaladowano dane historyczne (C:\Users\stasi\PycharmProjects\TenisPredictionModel\data\sample_data\atp_matches_2017.csv): 2784 meczow
Zaladowano dane historyczne (C:\Users\stasi\PycharmProjects\TenisPredictionModel\data\sample_data\atp_matches_2018.csv): 2787 meczow
Zaladowano dane historyczne (C:\Users\stasi\PycharmProjects\TenisPredictionModel\data\sample_data\atp_matches_2019.csv): 2658 meczow
Zaladowano dane historyczne (C:\Users\stasi\PycharmProjects\TenisPredictionModel\data\sample_data\atp_matches_2020.csv): 1359 meczow
Zaladowano dane historyczne (C:\Users\stasi\PycharmProjects\TenisPredictionModel\data\sample_data\atp_matches_2021.csv): 2608 meczow
Zaladowano dane historyczne (C:\Users\stasi\PycharmProjects\TenisPredictionModel\data\sample_data\atp_matches_2022.csv): 2731 meczow
Zaladowano dane historyczne (C:\Users\stasi\PycharmProjects\TenisPred


Po symetryzacji:
Walidacja:  1058 próbek (y=1: 529, y=0: 529)
Test:       1060 próbek (y=1: 530, y=0: 530)


Próbki treningowe dla walidacji krzyżowej: 3176
Fitting 5 folds for each of 50 candidates, totalling 250 fits



Najlepsze hiperparametry: {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 8, 'max_samples': 0.9, 'max_features': 'sqrt', 'max_depth': 10, 'bootstrap': True}
Najlepszy wynik CV: neg_log_loss=-0.6359 | accuracy=0.6386 | roc_auc=0.6899


Trening finalnego modelu na 3176 próbkach...



=== WYNIKI NA ZBIORZE WALIDACYJNYM ===
Accuracy: 0.6106

Classification Report:
                 precision    recall  f1-score   support

Gracz 2 wygrywa       0.61      0.62      0.61       529
Gracz 1 wygrywa       0.61      0.60      0.61       529

       accuracy                           0.61      1058
      macro avg       0.61      0.61      0.61      1058
   weighted avg       0.61      0.61      0.61      1058


Macierz pomyłek:
[[326 203]
 [209 320]]

=== FINALNE WYNIKI NA ZBIORZE TESTOWYM ===
Accuracy: 0.6604

Classification Report:
                 precision    recall  f1-score   support

Gracz 2 wygrywa       0.66      0.66      0.66       530
Gracz 1 wygrywa       0.66      0.66      0.66       530

       accuracy                           0.66      1060
      macro avg       0.66      0.66      0.66      1060
   weighted avg       0.66      0.66      0.66      1060


Macierz pomyłek:
[[351 179]
 [181 349]]

=== PRZEWIDYWANIE ZWYCIĘZCÓW MECZÓW ===

ACCURACY PRZEWIDYWAN

                 feature  importance
36         rank_pts_diff    0.092958
35             rank_diff    0.087177
5        p1_rank_pts_log    0.044265
10       p2_rank_pts_log    0.043584
9            p2_rank_log    0.041684
4            p1_rank_log    0.040063
34     p2_return_pts_won    0.033627
33     p1_return_pts_won    0.032871
32  p2_bp_faced_per_game    0.029844
31  p1_bp_faced_per_game    0.029361
37              age_diff    0.029022
27     p1_second_won_pct    0.027756
21            p1_df_rate    0.027186
26      p2_first_won_pct    0.027007
25      p1_first_won_pct    0.026882
28     p2_second_won_pct    0.026680
39             form_diff    0.026565
22            p2_df_rate    0.025257
24       p2_first_in_pct    0.024051
19           p1_ace_rate    0.023976
20           p2_ace_rate    0.023796
30        p2_bp_save_pct    0.023599
23       p1_first_in_pct    0.022402
6                 p1_age    0.021998
29        p1_bp_save_pct    0.021988
11                p2_age    0.021785
1

Jakosc kalibracji (test set):
  Raw RF       -> Brier=0.2174, log-loss=0.6246, ECE=0.0430
  Calibrated   -> Brier=0.2172, log-loss=0.6237, ECE=0.0367

Optymalny prog (walidacja): 0.50 -> val match-acc = 0.6106
Match accuracy po kalibracji + threshold tuning (test): 0.6566 (65.66%)
Porownanie: baseline (prog=0.5, raw)=0.6566 | tuned=0.6566 | delta=+0.0000

Reliability table (test, calibrated):
 bin_lower  bin_upper  count  avg_predicted  observed_frequency  calibration_error
       0.0        0.1      0            NaN                 NaN                NaN
       0.1        0.2     30       0.169610            0.133333           0.036277
       0.2        0.3    140       0.257378            0.250000           0.007378
       0.3        0.4    191       0.347948            0.382199          -0.034251
       0.4        0.5    171       0.449115            0.403509           0.045607
       0.5        0.6    177       0.552298            0.604520          -0.052222
       0.6        0.7  

## 2. Budowa tabeli do slicingu

Dla każdego meczu testowego potrzebujemy:
- **atrybutów meczu** (surface, round, best_of itd.) — to po nich będziemy slice'ować
- **wyniku predykcji** (`correct_prediction`, `p1_win_probability`) — to nasza metryka

Tabela buduje się merge'em po `match_id`. Slice'ujemy na poziomie meczów (nie symetryzowanych par), bo decyzja modelu jest jedna na mecz.

In [3]:
def build_handedness_matchup(row):
    hands = sorted([row["winner_hand"], row["loser_hand"]])
    return f"{hands[0]}-vs-{hands[1]}"

def build_bucketed_feature(series, bins, labels):
    return pd.cut(series, bins=bins, labels=labels, include_lowest=True, right=True).astype("object")

match_context = df_test_raw[[
    "match_id", "surface", "tourney_level", "best_of", "round",
    "winner_hand", "loser_hand", "winner_rank", "loser_rank",
    "winner_age", "loser_age", "w_form", "l_form",
]].copy()

match_context["handedness_matchup"] = match_context.apply(build_handedness_matchup, axis=1)
match_context["rank_gap_bucket"] = build_bucketed_feature(
    (match_context["winner_rank"] - match_context["loser_rank"]).abs(),
    bins=[-0.1, 10, 25, 50, 100, np.inf],
    labels=["0-10", "11-25", "26-50", "51-100", ">100"],
)
match_context["age_gap_bucket"] = build_bucketed_feature(
    (match_context["winner_age"] - match_context["loser_age"]).abs(),
    bins=[-0.1, 2, 5, 8, np.inf],
    labels=["0-2", "3-5", "6-8", ">8"],
)
match_context["form_gap_bucket"] = build_bucketed_feature(
    (match_context["w_form"] - match_context["l_form"]).abs(),
    bins=[-0.001, 0.10, 0.25, 0.40, 1.0],
    labels=["0.00-0.10", "0.10-0.25", "0.25-0.40", ">0.40"],
)

evaluation = winner_perspective[[
    "match_id", "correct_prediction", "p1_win_probability",
    "predicted_winner", "actual_winner",
]].copy()
evaluation["correct_prediction"] = evaluation["correct_prediction"].astype(int)

match_slice_frame = match_context.merge(evaluation, on="match_id", how="inner", validate="one_to_one")

# Sanity check: accuracy po joinie musi sie zgadzac z reported_match_accuracy
computed_accuracy = float(match_slice_frame["correct_prediction"].mean())
assert np.isclose(computed_accuracy, reported_match_accuracy), \
    f"Niespojnosc accuracy: {computed_accuracy:.4f} vs {reported_match_accuracy:.4f}"

print(f"Tabela do slicingu: {len(match_slice_frame)} wierszy, kolumny do slicingu = 8")
print(f"Sanity check: accuracy po joinie = {computed_accuracy:.4f} OK")

Tabela do slicingu: 530 wierszy, kolumny do slicingu = 8
Sanity check: accuracy po joinie = 0.6566 OK


## 3. Wilson Confidence Interval — dlaczego nie naiwne CI

Przy małym support (np. 5-20 meczów w slice'u) naiwne CI `p ± z·√(p(1-p)/n)` daje bzdurne granice:
- gdy p=0 lub p=1, naiwne CI to [0, 0] albo [1, 1] (nie da się ocenić niepewności)
- przy n=5 i p=0.4, naiwne CI sięga poza [0,1] albo jest absurdalnie wąskie

**Wilson** rozwiązuje to przez przesunięcie środka CI w stronę 0.5 — daje sensowne granice nawet przy n=5.

**Definicja istotności**: slice jest *statystycznie gorszy od średniej*, jeśli **górny brzeg jego CI < accuracy ogólne**. Wtedy mamy 95% pewność, że to nie szum.

In [4]:
def wilson_confidence_interval(successes, n, z=WILSON_Z):
    """95% CI dla proporcji wedlug Wilson score interval."""
    if n <= 0:
        return (0.0, 1.0)
    phat = successes / n
    z2 = z * z
    denom = 1.0 + z2 / n
    center = phat + z2 / (2.0 * n)
    margin = z * np.sqrt(phat * (1.0 - phat) / n + z2 / (4.0 * n * n))
    lower = max(0.0, (center - margin) / denom)
    upper = min(1.0, (center + margin) / denom)
    return float(lower), float(upper)

# Test: 4 sukcesy na 5 prob -> p=0.8, ale CI [0.376, 0.964] -- duza niepewnosc!
lower, upper = wilson_confidence_interval(4, 5)
print(f"4/5 (p=0.80): Wilson CI = [{lower:.3f}, {upper:.3f}]  -- szeroki, bo n male")

# Test: 80 sukcesow na 100 prob -> ta sama proporcja, ale duzo wezszy CI
lower, upper = wilson_confidence_interval(80, 100)
print(f"80/100 (p=0.80): Wilson CI = [{lower:.3f}, {upper:.3f}]  -- waski, bo n duze")

4/5 (p=0.80): Wilson CI = [0.376, 0.964]  -- szeroki, bo n male
80/100 (p=0.80): Wilson CI = [0.711, 0.867]  -- waski, bo n duze


## 4. Definicja `compute_model_slices`

Funkcja iteruje po wszystkich kombinacjach atrybutów (degree 1 i 2), grupuje mecze i liczy:
- **accuracy** + **error_rate**
- **Wilson CI** (lower/upper)
- flagę `statistically_below_overall` — czy slice jest istotnie gorszy od średniej
- **Brier Score** = mean((1 - p_prawdziwy_zwyciezca)²) — kara za błędne pewne predykcje
- **Log Loss** = -mean(log(p_prawdziwy_zwyciezca)) — standardowa metryka kalibracji
- **avg_true_winner_probability** — średnie p przypisane rzeczywistemu zwycięzcy

Min support = 5 chroni przed slice'ami zbyt małymi do interpretacji.

In [5]:
def slice_description(columns, values):
    return " & ".join(f"{c}={v}" for c, v in zip(columns, values))

def compute_model_slices(match_slice_frame, slice_columns, min_support=MIN_SUPPORT, max_degree=MAX_SLICE_DEGREE):
    overall_accuracy = float(match_slice_frame["correct_prediction"].mean())
    total = len(match_slice_frame)
    rows = []
    for degree in range(1, max_degree + 1):
        for combo in itertools.combinations(slice_columns, degree):
            grouped = match_slice_frame.groupby(list(combo), dropna=False, observed=True)
            for group_key, group_df in grouped:
                if not isinstance(group_key, tuple):
                    group_key = (group_key,)
                support = len(group_df)
                if support < min_support:
                    continue
                correct = int(group_df["correct_prediction"].sum())
                accuracy = correct / support
                proba = group_df["p1_win_probability"].astype(float)
                ci_lo, ci_hi = wilson_confidence_interval(correct, support)
                clipped = proba.clip(LOG_EPS, 1 - LOG_EPS)
                rows.append({
                    "slice_degree": degree,
                    "slice_definition": slice_description(combo, group_key),
                    "support": support,
                    "accuracy": accuracy,
                    "accuracy_gap_vs_overall": accuracy - overall_accuracy,
                    "ci_lower": ci_lo,
                    "ci_upper": ci_hi,
                    "sig_below": ci_hi < overall_accuracy,
                    "brier": float(((1.0 - proba) ** 2).mean()),
                    "log_loss": float(-np.log(clipped).mean()),
                    "avg_prob": float(proba.mean()),
                })
    return pd.DataFrame(rows).sort_values(["accuracy", "support"], ascending=[True, False]).reset_index(drop=True)

slice_columns = [
    "surface", "tourney_level", "best_of", "round",
    "handedness_matchup", "rank_gap_bucket", "age_gap_bucket", "form_gap_bucket",
]

slice_results = compute_model_slices(match_slice_frame, slice_columns)
overall_accuracy = float(match_slice_frame["correct_prediction"].mean())

print(f"Slice'ow wygenerowanych: {len(slice_results)}")
print(f"Accuracy ogolne: {overall_accuracy:.4f}")
print(f"Slice'ow istotnie ponizej sredniej: {int(slice_results['sig_below'].sum())}")

Slice'ow wygenerowanych: 349
Accuracy ogolne: 0.6566
Slice'ow istotnie ponizej sredniej: 5


## 5. Najsłabsze slice'y 1D

Jednowymiarowe slice'y to pojedyncze atrybuty — np. `best_of=5` albo `round=QF`. Pokazują najsłabsze podgrupy wziąwszy pod uwagę tylko jeden wymiar.

In [6]:
one_d = slice_results[slice_results["slice_degree"] == 1].copy()
worst_1d = one_d[one_d["accuracy_gap_vs_overall"] <= UNDERPERFORMANCE_GAP].head(TOP_N)
if worst_1d.empty:
    worst_1d = one_d.head(TOP_N)

print("=== NAJSLABSZE SLICE 1D ===")
print(worst_1d[["slice_definition", "support", "accuracy", "accuracy_gap_vs_overall", "ci_lower", "ci_upper", "sig_below", "brier"]].to_string(index=False))

=== NAJSLABSZE SLICE 1D ===
         slice_definition  support  accuracy  accuracy_gap_vs_overall  ci_lower  ci_upper  sig_below    brier
               round=R128       32  0.593750                -0.062854  0.422597  0.744806      False 0.238130
       age_gap_bucket=6-8      101  0.594059                -0.062544  0.496548  0.684678      False 0.250259
form_gap_bucket=0.00-0.10      221  0.597285                -0.059319  0.531494  0.659752      False 0.230866
     rank_gap_bucket=>100       80  0.600000                -0.056604  0.490453  0.700383      False 0.236155
    rank_gap_bucket=11-25      118  0.601695                -0.054909  0.511502  0.685475      False 0.238217


## 6. Najsłabsze slice'y 2D

Dwuwymiarowe slice'y łączą dwa atrybuty (np. `surface=Grass & round=QF`). Wyłapują **interakcje** — sytuacje, gdzie problem pojawia się tylko przy konkretnej kombinacji warunków.

In [7]:
two_d = slice_results[slice_results["slice_degree"] == 2].copy()
worst_2d = two_d[two_d["accuracy_gap_vs_overall"] <= UNDERPERFORMANCE_GAP].head(TOP_N)
if worst_2d.empty:
    worst_2d = two_d.head(TOP_N)

print("=== NAJSLABSZE SLICE 2D ===")
print(worst_2d[["slice_definition", "support", "accuracy", "accuracy_gap_vs_overall", "ci_lower", "ci_upper", "sig_below"]].to_string(index=False))

=== NAJSLABSZE SLICE 2D ===
                                 slice_definition  support  accuracy  accuracy_gap_vs_overall  ci_lower  ci_upper  sig_below
      tourney_level=F & form_gap_bucket=0.00-0.10        6  0.333333                -0.323270  0.096769  0.700012      False
               round=R128 & rank_gap_bucket=26-50        6  0.333333                -0.323270  0.096769  0.700012      False
           tourney_level=M & rank_gap_bucket=>100       19  0.368421                -0.288183  0.191493  0.589608       True
           round=R128 & handedness_matchup=L-vs-R        8  0.375000                -0.281604  0.136842  0.694262      False
               round=R128 & rank_gap_bucket=11-25        8  0.375000                -0.281604  0.136842  0.694262      False
rank_gap_bucket=11-25 & form_gap_bucket=0.25-0.40       23  0.391304                -0.265299  0.221574  0.592148       True
                 round=SF & rank_gap_bucket=26-50        5  0.400000                -0.256604  0.

## 7. Najlepsze slice'y dla kontrastu

Pokazujemy też najmocniejsze podgrupy — to pomaga zrozumieć **gdzie model dobrze działa**. Jeśli model świetnie radzi sobie na rank_gap=0-10 (faworyt vs faworyt), ale słabo na rank_gap >100 (faworyt vs underdog), to wskazówka, że rankingiem dominuje predykcje.

In [8]:
best = slice_results.sort_values(["accuracy", "support"], ascending=[False, False]).head(8)
print("=== NAJLEPSZE SLICE ===")
print(best[["slice_definition", "support", "accuracy", "accuracy_gap_vs_overall"]].to_string(index=False))

=== NAJLEPSZE SLICE ===
                           slice_definition  support  accuracy  accuracy_gap_vs_overall
      best_of=5 & handedness_matchup=L-vs-R        9  1.000000                 0.343396
        round=F & form_gap_bucket=0.10-0.25        7  1.000000                 0.343396
       round=QF & form_gap_bucket=0.25-0.40        7  1.000000                 0.343396
          round=R128 & rank_gap_bucket=0-10        6  1.000000                 0.343396
tourney_level=G & handedness_matchup=L-vs-R        5  1.000000                 0.343396
              round=SF & age_gap_bucket=0-2       11  0.909091                 0.252487
              round=R16 & age_gap_bucket=>8       21  0.904762                 0.248158
tourney_level=F & form_gap_bucket=0.10-0.25        9  0.888889                 0.232285


## 8. Wnioski

**Wyniki z bieżącego uruchomienia (RANDOM_STATE=42, sezon 2025, metryka symetryczna)**:
- Match accuracy ogólne: **65.66%** (348 z 530 meczów testowych)
- Slice'ów wygenerowanych (degree 1 + 2, support >= 5): **349**
- Slice'ów istotnie poniżej średniej (Wilson upper CI < overall): **5**

Model nie myli się losowo -- istnieją konkretne, semantyczne podgrupy meczów, w których accuracy jest niższe niż średnio. Najsłabsze miejsca w tym uruchomieniu:

1. **round=R128** (pierwsze rundy Grand Slamu) -- support 32, accuracy 59.4%. Duża rozpiętość poziomów graczy + nieprzewidywalność wczesnych rund; to samo słabe miejsce co w poprzednich uruchomieniach.
2. **Wyrównane mecze**: form_gap 0.00-0.10 (support 221, accuracy 59.7%) i rank_gap 11-25 (support 118, accuracy 60.2%) -- gdy gracze są zbliżeni, cechy różnicowe niewiele wnoszą.
3. **tourney_level=M & rank_gap>100** (support 19, accuracy 36.8%) oraz **rank_gap=11-25 & form_gap=0.25-0.40** (support 23, accuracy 39.1%) -- jedyne większe slice'y z flagą `sig_below=True`; na Mastersach przy dużych różnicach rankingowych model za bardzo ufa faworytowi.
4. **best_of=5 NIE jest już najsłabszym slicem** -- w przeciwieństwie do historycznego runa na sezonie 2024 (Bo5 ~48-52%); na 2025 wśród najlepszych slice'ów jest wręcz `best_of=5 & L-vs-R` (9/9 trafień). Słabe miejsca przesuwają się między sezonami -- to argument przeciwko projektowaniu cech pod slice'y z jednego sezonu.

Wilson CI pomaga oddzielić prawdziwą słabość od szumu: na 349 slice'ów tylko 5 jest istotnie poniżej średniej, a większość „spektakularnych" spadków ma support 5-10 i bardzo szeroki przedział ufności.

Kolejny krok: slice-aware warianty modelu -- notebooki `TPM_Experiment_SliceAware*`. Wyniki bieżące (sezon 2025, metryka symetryczna): BestOf5 v1 **+0.94 p.p.**, SliceAware **-0.38 p.p.**, QFServe v3 **-2.08 p.p.** Walk-forward 2020-2025 nie potwierdza istotnej poprawy żadnego wariantu (docs/WYNIKI_SPRINTOW.md, docs/PODSUMOWANIE_KONCOWE.md). Historyczne „+2.37/+2.20 p.p." z sezonu 2024 było w dużej mierze artefaktem starej, jednostronnej metryki.